In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('sentiment_dataset_1000 (1).csv')

In [ ]:
df.head(3)

,text,label
0,The app was disappointing and a waste of money.,0
1,The game experience was frustrating and bad.,0
2,"I really hated this hotel, it was terrible.",0


In [ ]:
df.sample(3)

,text,label
904,"Wonderful service, I enjoyed it a lot.",1
776,The meal was disappointing and a waste of money.,0
693,"This product broke after one use, very poor qu...",0


In [ ]:
df.shape

(1000, 2)

In [ ]:
texts = df['text'].values

In [ ]:
labels = df['label'].values

In [ ]:
vocab_size = 80
max_len = 20

In [ ]:
tokenizer = Tokenizer(num_words=vocab_size,oov_token='nasir')

In [ ]:
tokenizer.fit_on_texts(texts)

In [ ]:
tokenizer.word_counts

OrderedDict([('the', 301),
             ('app', 113),
             ('was', 401),
             ('disappointing', 60),
             ('and', 338),
             ('a', 116),
             ('waste', 60),
             ('of', 60),
             ('money', 60),
             ('game', 108),
             ('experience', 207),
             ('frustrating', 46),
             ('bad', 46),
             ('i', 409),
             ('really', 96),
             ('hated', 43),
             ('this', 455),
             ('hotel', 90),
             ('it', 296),
             ('terrible', 43),
             ('wouldn’t', 52),
             ('recommend', 93),
             ('movie', 96),
             ('not', 52),
             ('worth', 95),
             ('i’m', 121),
             ('happy', 61),
             ('chose', 61),
             ('meal', 91),
             ('broke', 55),
             ('after', 55),
             ('one', 55),
             ('use', 108),
             ('very', 156),
             ('poor', 55),
             (

In [ ]:
sequences = tokenizer.texts_to_sequences(texts)

In [ ]:
texts[0]

'The app was disappointing and a waste of money.'

In [ ]:
sequences[1]

[6, 15, 8, 4, 56, 5, 57]

In [ ]:
padded = pad_sequences(sequences,maxlen=max_len,padding='post',truncating='post')

In [ ]:
padded[1]

array([ 6, 15,  8,  4, 56,  5, 57,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0], dtype=int32)

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(padded,labels,test_size=0.2,
                                                 random_state=42)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=vocab_size,output_dim=64,input_length=max_len),
    tf.keras.layers.LSTM(128,return_sequences=False),
    tf.keras.layers.Dense(64,activation='relu'),
    tf.keras.layers.Dense(1,activation='sigmoid')
])

In [ ]:
padded[:5]

array([[ 6, 13,  4, 31,  5, 12, 32, 33, 34,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0],
       [ 6, 15,  8,  4, 56,  5, 57,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0],
       [ 3, 19, 63,  2, 27,  7,  4, 64,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0],
       [ 3, 52, 23,  2, 20, 53, 21,  7,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0],
       [11, 29,  3, 30,  2, 24,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0]], dtype=int32)

In [ ]:
vs = len(tokenizer.word_index) + 1
vs

80

In [ ]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (32, 20, 64)           │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (32, 128)              │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (32, 64)               │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (32, 1)                │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 112,257 (438.50 KB)

 Trainable params: 112,257 (438.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train,y_train,
    validation_data=(X_test,y_test),
    epochs=5,
    batch_size=32
)

Epoch 1/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 0.5523 - loss: 0.6862 - val_accuracy: 0.9750 - val_loss: 0.2551
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9935 - loss: 0.1118 - val_accuracy: 1.0000 - val_loss: 0.0067
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9844 - loss: 0.0504 - val_accuracy: 1.0000 - val_loss: 0.0122
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 1.0000 - loss: 0.0051 - val_accuracy: 1.0000 - val_loss: 6.8808e-04
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 1.0000 - loss: 5.6347e-04 - val_accuracy: 1.0000 - val_loss: 3.3178e-04


In [ ]:
loss, acc = model.evaluate(X_test,y_test)
print('Test Accuracy:', acc)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 3.2786e-04
Test Accuracy: 1.0


In [ ]:
def predict_sentiment(sentence):
  seq = tokenizer.texts_to_sequences([sentence])
  pad = pad_sequences(seq,maxlen=max_len,padding='post',truncating='post')
  pred = model.predict(pad)
  if pred > 0.5:
    print('Positive Sentiment')
  else:
    print('Negative Sentiment')

In [ ]:
print(predict_sentiment('i like the product'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 526ms/step
Positive Sentiment
None


In [ ]:
print(predict_sentiment('i love RNN'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Positive Sentiment
None


In [ ]:
print(predict_sentiment('product was good'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Positive Sentiment
None


In [ ]:
print(predict_sentiment('product was bad'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Positive Sentiment
None


In [ ]:
print(predict_sentiment('product was worst'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Positive Sentiment
None


In [ ]:
print(predict_sentiment('product was best'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Positive Sentiment
None


In [ ]:
print(predict_sentiment('i am happy with this product'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Positive Sentiment
None
